# DSA 4060 – Recommender Systems
## Week 3 Practical Lab: Building a Simple Content-Based Recommender

**Student ID:** 670444

### Lab objective

Build a content-based recommender that suggests movies to a user based on the
**structured features** (binary genre flags) of movies they have already rated highly.

This lab deliberately stays with structured features and a weighted dot product.
**TF-IDF and cosine similarity are not used here — they are Week 4 topics.**

### Workflow implemented in this notebook

```
Item Metadata -> Item Profiles -> User Ratings -> User Profile
   -> Candidate Scores -> Remove Seen Items -> Rank Items -> Top-N Recommendations
```

### Learning outcomes covered

- Represent movies using structured item features
- Create and interpret an item-feature matrix
- Build a user preference profile from ratings
- Score and rank candidate movies
- Remove movies already watched by the user
- Generate Top-N recommendations
- Explain the limitations of a basic content-based recommender

## 1. Import Libraries

*(Lab Step 2)*

In [1]:
import pandas as pd

pd.set_option("display.width", 120)
print("pandas version:", pd.__version__)

pandas version: 2.3.3


## 2. Create the Movie Dataset

*(Lab Step 3)*

Each row is one movie. The five genre columns are **binary structured features**:
`1` = feature present, `0` = feature absent.

In [2]:
movies = pd.DataFrame({
    "movie_id": [1,2,3,4,5,6,7,8,9,10],
    "title": [
        "Galaxy Mission","Love in Nairobi","Final Battle","Family Weekend",
        "Future World","Broken Hearts","City Detectives","Space Warriors",
        "Wedding Weekend","Hidden Truth"
    ],
    "Action":  [1,0,1,0,1,0,1,1,0,0],
    "Comedy":  [0,0,0,1,0,0,0,0,1,0],
    "Drama":   [0,1,0,0,0,1,1,0,0,1],
    "Romance": [0,1,0,0,0,1,0,0,1,0],
    "SciFi":   [1,0,0,0,1,0,0,1,0,0]
})

movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


### Save the dataset to `movies.csv`

`movies.csv` is a required file in the submitted repository, so the DataFrame created
above is written to disk here. The cell below then reloads it, which confirms the CSV
and the in-memory DataFrame hold the same data.

In [3]:
movies.to_csv("movies.csv", index=False)

movies = pd.read_csv("movies.csv")
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


## 3. Understand and Explore the Dataset

*(Lab Steps 4 and 5)*

| Column | Meaning |
|---|---|
| `movie_id` | Unique movie identifier |
| `title` | Movie title |
| `Action` / `Comedy` / `Drama` / `Romance` / `SciFi` | Binary structured features |

**Feature encoding:** `1` = feature present, `0` = feature absent.
Example: *Galaxy Mission* has `Action = 1` and `SciFi = 1`, so its item profile is `[1, 0, 0, 0, 1]`.

In [4]:
movies.head()

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1


In [5]:
movies.shape

(10, 7)

In [6]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  10 non-null     int64 
 1   title     10 non-null     object
 2   Action    10 non-null     int64 
 3   Comedy    10 non-null     int64 
 4   Drama     10 non-null     int64 
 5   Romance   10 non-null     int64 
 6   SciFi     10 non-null     int64 
dtypes: int64(6), object(1)
memory usage: 688.0+ bytes


### Step 5 answers

**1. How many movies are in the dataset?**
There are **10 movies** — `movies.shape` returns `(10, 7)`, so 10 rows.

**2. How many genre features exist?**
There are **5 genre features**: `Action`, `Comedy`, `Drama`, `Romance`, `SciFi`.
The remaining two columns (`movie_id`, `title`) are identifiers, not features.

**3. What does one row represent?**
One row represents **a single movie together with its complete item profile** — its id,
its title, and the presence or absence of each of the five genres. A row is the unit the
recommender scores.

## 4. Identify the Structured Features

*(Lab Step 6)*

Separating the feature columns from the identifier columns gives the **item-feature
matrix** — the numeric block the recommender actually does arithmetic on.

In [7]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]

movie_features = movies[features]

movie_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,1,0,0,0,1
5,0,0,1,1,0
6,1,0,1,0,0
7,1,0,0,0,1
8,0,1,0,1,0
9,0,0,1,0,0


## 5. Inspect One Item Profile

*(Lab Step 7)*

In [8]:
movies.loc[
    movies["title"] == "Galaxy Mission"
]

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1


### Step 7 answer — what the 1s and 0s mean

*Galaxy Mission* has `Action = 1` and `SciFi = 1`, and `0` in `Comedy`, `Drama` and
`Romance`. The `1`s say the movie **does belong** to those two genres and the `0`s say it
**does not** belong to the other three, so its item profile is `[1, 0, 0, 0, 1]`.

This vector is the movie's entire representation as far as the model is concerned — the
recommender knows nothing about the film beyond these five flags.

## 6. Create User Ratings (User A)

*(Lab Steps 8, 9 and 10)*

User A has rated four movies:

| Movie | Rating |
|---|---|
| Galaxy Mission | 5 |
| Love in Nairobi | 2 |
| Final Battle | 4 |
| Future World | 5 |

In [9]:
ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating": [5, 2, 4, 5]
})

ratings

,movie_id,rating
0,1,5
1,2,2
2,3,4
3,5,5


### Join the ratings with the movie features

*(Lab Step 9)*

In [10]:
user_movies = ratings.merge(
    movies,
    on="movie_id"
)

user_movies

,movie_id,rating,title,Action,Comedy,Drama,Romance,SciFi
0,1,5,Galaxy Mission,1,0,0,0,1
1,2,2,Love in Nairobi,0,0,1,1,0
2,3,4,Final Battle,1,0,0,0,0
3,5,5,Future World,1,0,0,0,1


### Examine the user's history

*(Lab Step 10)*

In [11]:
user_movies[
    ["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]
]

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,Love in Nairobi,2,0,0,1,1,0
2,Final Battle,4,1,0,0,0,0
3,Future World,5,1,0,0,0,1


### Step 10 observation — what does this user appear to prefer?

*(Written before any calculation.)*

The three movies User A rated highly are *Galaxy Mission* (5), *Future World* (5) and
*Final Battle* (4), and all three are **Action**; two of them are also **SciFi**. The only
movie they rated poorly, *Love in Nairobi* (2), is the one Drama/Romance title in their
history.

So by inspection this looks like an **action and science-fiction viewer who dislikes
romantic drama**. I expect the profile to be dominated by `Action` and `SciFi`.

## 7. Build the User Profile

*(Lab Steps 11 to 15)*

### Select the rated features

In [12]:
rated_features = user_movies[features]
user_ratings = user_movies["rating"]

rated_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,1,0,0,0,1


In [13]:
user_ratings

0    5
1    2
2    4
3    5
Name: rating, dtype: int64

### Weight the features using the ratings

*(Lab Step 12)*

Each movie's feature row is multiplied by the rating the user gave it. A highly rated
movie therefore contributes more strongly to the preference profile than a poorly rated one.

In [14]:
weighted_features = rated_features.mul(
    user_ratings,
    axis=0
)

weighted_features

,Action,Comedy,Drama,Romance,SciFi
0,5,0,0,0,5
1,0,0,2,2,0
2,4,0,0,0,0
3,5,0,0,0,5


### Sum the weighted features into the profile

*(Lab Step 13)*

Summing **down each column** collapses the user's whole history into one vector with a
score per genre.

In [15]:
user_profile = weighted_features.sum()

user_profile

Action     14
Comedy      0
Drama       2
Romance     2
SciFi      10
dtype: int64

### Interpret the profile

*(Lab Step 14)*

In [16]:
user_profile.sort_values(
    ascending=False
)

Action     14
SciFi      10
Drama       2
Romance     2
Comedy      0
dtype: int64

### Step 14 answer — which two genres are strongest, and why?

The two strongest genres are **Action (14)** and **SciFi (10)**.

Reading the profile scores:

- **Action = 14** — three of the four rated movies are Action, and they carry the high
  ratings: `5 (Galaxy Mission) + 4 (Final Battle) + 5 (Future World) = 14`.
- **SciFi = 10** — two Action movies are also SciFi, both rated 5: `5 + 5 = 10`.
- **Drama = 2** and **Romance = 2** — these come only from *Love in Nairobi*, which the
  user rated 2, so they are heavily penalised by the low weight.
- **Comedy = 0** — the user has not rated a single comedy, so the model has no evidence
  about comedy at all.

This confirms the observation made in Step 10 from inspection alone.

### Normalize the profile

*(Lab Step 15)*

Dividing by the total converts the raw sums into **proportions** that add to 1, which
makes the relative strength of each genre easier to read and keeps scores on a comparable
scale between users who have rated different numbers of movies.

In [17]:
user_profile_normalized = (
    user_profile / user_profile.sum()
)

user_profile_normalized

Action     0.500000
Comedy     0.000000
Drama      0.071429
Romance    0.071429
SciFi      0.357143
dtype: float64

In [18]:
# The normalized profile sums to 1 — a useful sanity check.
user_profile_normalized.sum()

np.float64(1.0)

After normalization the profile reads as: **50.0% Action, 35.7% SciFi, 7.1% Drama,
7.1% Romance, 0% Comedy**. Half of this user's evidence-weighted preference is Action.

## 8. Calculate Recommendation Scores

*(Lab Steps 16 and 17)*

Every movie's feature vector is multiplied element-wise by the normalized user profile
and summed across the row. This is a **weighted dot product**: a movie scores points for
each genre it has, weighted by how much the user likes that genre.

In [19]:
movies["recommendation_score"] = (
    movie_features
    .mul(user_profile_normalized)
    .sum(axis=1)
)

movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
1,Love in Nairobi,0.142857
2,Final Battle,0.500000
3,Family Weekend,0.000000
4,Future World,0.857143
5,Broken Hearts,0.142857
6,City Detectives,0.571429
7,Space Warriors,0.857143
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429


### Rank the movies

*(Lab Step 17)*

In [20]:
ranked_movies = movies.sort_values(
    "recommendation_score",
    ascending=False
)

ranked_movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
4,Future World,0.857143
7,Space Warriors,0.857143
6,City Detectives,0.571429
2,Final Battle,0.500000
1,Love in Nairobi,0.142857
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


A higher recommendation score means the movie contains more of the features this user
prefers. Note that the movies the user has **already rated** are still in this ranking —
*Galaxy Mission* and *Future World* sit at the top with 0.857, which is expected, since
the profile was built from them. They are filtered out next.

## 9. Remove Movies Already Watched

*(Lab Step 18)*

Recommending a movie the user has already seen and rated is useless, so candidates are
restricted to unseen `movie_id`s.

In [21]:
watched_movies = ratings["movie_id"]

recommendations = ranked_movies[
    ~ranked_movies["movie_id"].isin(watched_movies)
]

recommendations[["title", "recommendation_score"]]

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


## 10. Generate the Top 3 Recommendations

*(Lab Step 19)*

In [22]:
top_3 = recommendations[
    ["title", "recommendation_score"]
].head(3)

top_3

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857


### Step 20 answer — why did the model recommend each of these?

User A's strongest feature preferences are **Action (0.500)** and **SciFi (0.357)**.

1. **Space Warriors — 0.857.** It is both Action and SciFi, so it collects the user's two
   largest profile weights at once (`0.500 + 0.357`) — an exact feature match for the pair
   of genres this user rates most highly.
2. **City Detectives — 0.571.** It is Action, which alone supplies the single biggest
   weight in the profile (`0.500`), plus a small amount of Drama (`0.071`); it is
   recommended mainly on the strength of the Action match.
3. **Broken Hearts — 0.143.** It only matches the user's two *weakest* non-zero
   preferences, Drama and Romance (`0.071 + 0.071`). It appears third not because it fits
   the user well, but because the remaining candidates fit even less — *Wedding Weekend*
   and *Hidden Truth* score 0.071 and *Family Weekend* scores 0.000, as it is pure Comedy
   and the user has no Comedy evidence at all.

This third recommendation is already a warning sign about the model: with only ten movies
and five features, the list runs out of genuinely good matches after two items.

## 11. Build a Reusable Function

*(Lab Step 21)*

The whole pipeline above is wrapped into one function so it can be re-run for any user.
It uses `movies.copy()` so that scoring one user never mutates the shared movie table.

In [23]:
def recommend_movies(movies, ratings, features, n=3):

    user_movies = ratings.merge(movies, on="movie_id")

    rated_features = user_movies[features]

    weighted_features = rated_features.mul(
        user_movies["rating"],
        axis=0
    )

    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()

    scores = (
        movies[features]
        .mul(user_profile)
        .sum(axis=1)
    )

    results = movies.copy()
    results["recommendation_score"] = scores

    watched = ratings["movie_id"]

    results = results[
        ~results["movie_id"].isin(watched)
    ]

    return results.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)

### Test the function against the manual result

In [24]:
recommend_movies(
    movies,
    ratings,
    features
)[["title", "recommendation_score"]]

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857


This reproduces the Top 3 computed step by step above — *Space Warriors* (0.857),
*City Detectives* (0.571), *Broken Hearts* (0.143) — which confirms the function is
equivalent to the manual pipeline.

### A small helper for inspecting profiles

The function returns only recommendations, but several questions below ask *why* a user
received them, which needs the profile itself. This helper exposes it.

In [25]:
def user_profile_for(movies, ratings, features):
    """Return the normalized preference profile for one user."""
    user_movies = ratings.merge(movies, on="movie_id")
    profile = user_movies[features].mul(user_movies["rating"], axis=0).sum()
    return (profile / profile.sum()).sort_values(ascending=False)

## 12. Test a Second User (User B)

*(Lab Step 22)*

In [26]:
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating": [5, 4, 5, 4]
})

recommend_movies(
    movies,
    user_b_ratings,
    features
)[["title", "recommendation_score"]]

,title,recommendation_score
6,City Detectives,0.3125
9,Hidden Truth,0.3125
0,Galaxy Mission,0.0000


### Compare the two users' profiles side by side

In [27]:
comparison = pd.DataFrame({
    "User A": user_profile_for(movies, ratings, features),
    "User B": user_profile_for(movies, user_b_ratings, features)
}).fillna(0)

comparison

,User A,User B
Action,0.500000,0.0000
Comedy,0.000000,0.2500
Drama,0.071429,0.3125
Romance,0.071429,0.4375
SciFi,0.357143,0.0000


### Step 22 answer — did User A and User B get the same recommendations?

**No, and the two lists share almost nothing.**

The profiles are close to opposites:

| Genre | User A | User B |
|---|---|---|
| Action | **0.500** | 0.000 |
| SciFi | **0.357** | 0.000 |
| Romance | 0.071 | **0.438** |
| Drama | 0.071 | **0.313** |
| Comedy | 0.000 | **0.250** |

**Why they differ:** the recommendations are driven entirely by the user profile, and the
profile is built from *which* movies each user rated and *how highly*. User A rated Action
and SciFi films highly, so those genres dominate their profile. User B rated
Drama/Romance/Comedy films highly (*Love in Nairobi* 5, *Broken Hearts* 5, *Family
Weekend* 4, *Wedding Weekend* 4) and never rated an Action or SciFi film at all, so both
of those weights are exactly **0**.

Because User B's Action and SciFi weights are zero, every pure Action/SciFi film —
*Galaxy Mission*, *Final Battle*, *Future World*, *Space Warriors* — scores **0.000** for
them, even though *Space Warriors* was User A's single best recommendation.

**One thing worth noticing:** User B's top two recommendations, *City Detectives* and
*Hidden Truth*, **tie at 0.3125**. Both match on Drama and nothing else. The model has no
way to break that tie, so the order between them is decided by the sort, not by evidence —
a real weakness when features are this coarse.

## 13. Thinking Exercise — User C

*(Lab Step 23)*

Suppose User C has watched only *Galaxy Mission* and rated it 5 stars.

In [28]:
user_c_ratings = pd.DataFrame({
    "movie_id": [1],
    "rating": [5]
})

print("User C profile:")
print(user_profile_for(movies, user_c_ratings, features))

User C profile:
Action     0.5
SciFi      0.5
Comedy     0.0
Drama      0.0
Romance    0.0
dtype: float64


In [29]:
recommend_movies(
    movies,
    user_c_ratings,
    features,
    n=9
)[["title", "recommendation_score"]]

,title,recommendation_score
4,Future World,1.0
7,Space Warriors,1.0
2,Final Battle,0.5
6,City Detectives,0.5
1,Love in Nairobi,0.0
3,Family Weekend,0.0
5,Broken Hearts,0.0
8,Wedding Weekend,0.0
9,Hidden Truth,0.0


### Step 23 answers

**4. What will the model learn about User C?**

Very little, and it will be dangerously overconfident about it. *Galaxy Mission* is
`[1, 0, 0, 0, 1]`, so the only evidence available is "this person likes one Action/SciFi
film". After normalization the profile is **Action 0.5, SciFi 0.5, everything else 0.0**.
Note that the rating value is irrelevant here: with a single rated movie, normalization
divides it out, so a 5-star and a 1-star rating produce the *identical* profile. The model
cannot represent dislike at all from one rating.

**5. Which genres will dominate the profile?**

**Action and SciFi, at exactly 50% each** — and they will be the *only* non-zero genres.
Consequently *Future World* and *Space Warriors* both score a perfect **1.0**, while five
of the nine candidates score **0.0** and are completely unrankable among themselves.

**6. What could go wrong with the recommendations?**

- **Over-specialisation / a filter bubble.** Every recommendation will be Action or SciFi.
  The user will never be shown a comedy, drama or romance, so the model can never discover
  that they might enjoy one.
- **Zero-score ties.** Five movies score 0.0 and two score 1.0. The model cannot
  meaningfully order either group, so ranking within them is arbitrary.
- **A spurious genre link.** The model concludes Action and SciFi are *equally* important,
  but it has no way to tell whether the user liked the film for its action, its sci-fi, or
  something it cannot see at all — the director, the lead actor, the soundtrack.
- **No sense of confidence.** A profile built from one rating is presented with exactly the
  same authority as one built from a hundred. Nothing in the output signals that these
  recommendations rest on a single data point.

**7. Is one rating enough to understand a user's preferences?**

**No.** One rating identifies at most a couple of genres the user tolerated once; it cannot
separate a genuine preference from a coincidence, it cannot capture dislikes, and it cannot
express how strongly any preference is held. This is the classic **new-user (user) cold-start
problem**. A usable profile needs ratings spread across several *different* feature
combinations — including some low ratings, which are what let the model learn what to avoid.

## 14. Add My Unique Movie

*(Lab Step 24)*

Every student must add one unique movie with their own title and feature combination.

My chosen item is **"Money Heist"** (*La Casa de Papel*, 2017) — my favourite title.

**Feature justification.** Money Heist is a heist thriller, so `Action = 1` (the Royal Mint
siege, the police standoffs, the escapes). It is heavily character-driven, with the crew's
backstories and the hostage negotiations carrying most of the runtime, so `Drama = 1`. It
also runs several sustained romance subplots — Tokyo and Rio, the Professor and Raquel,
Denver and Mónica — so `Romance = 1`. Nothing in it is comedic enough to justify `Comedy`
and nothing is speculative enough to justify `SciFi`, so both stay `0`.

Its profile is therefore `[1, 0, 1, 1, 0]`. No existing movie in the dataset uses this
combination, so it adds a genuinely new item profile rather than duplicating one already
present — the cell below verifies that programmatically rather than by eye.

**One honest caveat:** Money Heist is a multi-season television series, not a feature film.
The dataset has no column for item format, so the model cannot tell the difference and
scores it exactly as it would a 90-minute movie. I return to this in the Limitations.

In [30]:
# Step 16 attached a 'recommendation_score' column (specific to User A) to `movies`.
# It is dropped before appending, so the new row does not pick up a NaN score and
# the saved CSV keeps only real movie metadata.
movies = movies.drop(columns=["recommendation_score"], errors="ignore")

new_movie = {
    "movie_id": 11,
    "title": "Money Heist",
    "Action": 1,
    "Comedy": 0,
    "Drama": 1,
    "Romance": 1,
    "SciFi": 0
}

movies = pd.concat(
    [movies, pd.DataFrame([new_movie])],
    ignore_index=True
)

movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


In [31]:
# Confirm the new feature combination is unique in the dataset.
duplicate_profiles = movies[features].duplicated(keep=False)
movies.loc[movies["movie_id"] == 11, features].apply(tuple, axis=1).iloc[0], \
    bool(duplicate_profiles.iloc[-1])

((1, 0, 1, 1, 0), False)

In [32]:
# Persist the extended dataset (11 movies) as the submitted movies.csv.
movies.to_csv("movies.csv", index=False)
movies.shape

(11, 7)

## 15. Create My Own User

*(Lab Step 25)*

My user rates four movies, including my unique movie, and receives **Top 5**
recommendations.

| Movie | Rating |
|---|---|
| Galaxy Mission (Action, SciFi) | 5 |
| City Detectives (Action, Drama) | 4 |
| Hidden Truth (Drama) | 4 |
| Money Heist (Action, Drama, Romance) | 5 |

In [33]:
my_user = pd.DataFrame({
    "movie_id": [1, 7, 10, 11],
    "rating": [5, 4, 4, 5]
})

my_user.merge(movies, on="movie_id")[
    ["title", "rating"] + features
]

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,City Detectives,4,1,0,1,0,0
2,Hidden Truth,4,0,0,1,0,0
3,Money Heist,5,1,0,1,1,0


In [34]:
my_profile = user_profile_for(movies, my_user, features)
my_profile

Action     0.378378
Drama      0.351351
Romance    0.135135
SciFi      0.135135
Comedy     0.000000
dtype: float64

In [35]:
my_top_5 = recommend_movies(
    movies,
    my_user,
    features,
    n=5
)

my_top_5[["title", "recommendation_score"]]

,title,recommendation_score
4,Future World,0.513514
7,Space Warriors,0.513514
1,Love in Nairobi,0.486486
5,Broken Hearts,0.486486
2,Final Battle,0.378378


## 16. Interpretation

*(Lab Step 26)*

**8. What are my user's two strongest preferences?**

**Action (0.378)** and **Drama (0.351)**, with **Romance and SciFi tied at 0.135** behind them.

Tracing the raw weighted sums before normalization makes the shape clear:

| Genre | Raw sum | Where it came from |
|---|---|---|
| Action | **14** | Galaxy Mission (5) + City Detectives (4) + Money Heist (5) |
| Drama | **13** | City Detectives (4) + Hidden Truth (4) + Money Heist (5) |
| Romance | 5 | Money Heist (5) only |
| SciFi | 5 | Galaxy Mission (5) only |
| Comedy | 0 | no rated comedy at all |

Action and Drama dominate because each is backed by *three* rated movies, while Romance and
SciFi each rest on a single film. Adding Money Heist is what introduced Romance into the
profile at all — without it that weight would be 0.

**9. Which three movies received the highest recommendation scores?**

| Rank | Movie | Score | Matching features |
|---|---|---|---|
| 1 | Future World | 0.514 | Action + SciFi |
| 2 | Space Warriors | 0.514 | Action + SciFi |
| 3 | Love in Nairobi | 0.486 | Drama + Romance |

*Broken Hearts* also scores **0.486**, tying with *Love in Nairobi* for third.

**10. Why did the top movie rank first?**

*Future World* is both **Action** and **SciFi**, so it collects the profile's largest weight
plus the SciFi weight: `0.378 + 0.135 = 0.514`.

Strictly, though, it did **not** rank first on merit. *Space Warriors* has the identical
profile `[1, 0, 0, 0, 1]` and therefore the identical score of 0.514. The two are tied, and
`sort_values` simply happened to return *Future World* first. The model has no basis
whatsoever for preferring one over the other.

**11. Did any recommendation surprise me?**

Three things did.

First, **four of my five recommendations sit in tied pairs** — ranks 1–2 tie at 0.514 and
ranks 3–4 tie at 0.486. With five binary features there are very few distinct scores
available, so ties are not an edge case here; they are the normal outcome, and most of the
ranking ends up decided by pandas' sort order rather than by evidence.

Second, **the gap between the top pair and the next pair is only 0.027.** An Action/SciFi
film (0.514) barely outranks a Drama/Romance film (0.486), even though my profile's
Action + Drama weights (0.378 + 0.351) are much larger than its SciFi + Romance weights
(0.135 + 0.135). That margin is thin enough that a single different rating anywhere in my
history would reorder the whole list.

Third, **Drama is my second-strongest preference (0.351), yet neither of my top two
recommendations contains any Drama at all.** They win on Action plus a weak SciFi signal
worth only 0.135. *City Detectives*, which matches both of my strongest preferences
(Action + Drama), would score 0.730 — higher than anything that actually appears in the
list — but it is absent because I had already rated it, so it is correctly filtered out as
watched. The model rewards
a movie for *how much total preference weight it covers*, not for whether it matches the
preferences I hold most strongly.

**12. What important information is the model ignoring?**

- **Everything except five genre flags** — no actors, directors, release year, language,
  runtime, plot description, budget or certification.
- **Item format.** Money Heist is a multi-season TV series sitting in a table of feature
  films, and nothing in the data marks that. The model would happily recommend a 15-hour
  commitment to someone who just wanted a film for the evening.
- **Degree of membership.** A film that is 90% action and one with a single chase scene both
  get `Action = 1`. The features are binary, so intensity is lost.
- **Other users entirely.** There is no collaborative signal — the model never asks what
  people with similar taste enjoyed, so it can never recommend something outside the genres
  I have already rated.
- **Popularity, quality and recency.** A badly reviewed Action film outranks a universally
  acclaimed comedy for my user, because quality is not a feature.
- **Explicit dislike.** Low ratings still add *positive* weight, just less of it. Rating a
  romance 1 star makes Romance slightly *more* attractive to the model, never less — there
  are no negative weights.
- **Context.** Who the user is watching with, their mood, the time of day, whether they want
  something familiar or something new.

## 17. Limitations

*(Lab Step 27)*

Six limitations of this recommender, the first three being the most serious.

**1. Only five binary features — so scores collide constantly.**
Five binary genres allow only a small number of distinct score values, which is why my own
user's Top 5 contained two tied pairs and User B's top two tied exactly at 0.3125. When
scores tie, the ranking is decided by pandas' sort order rather than by evidence. Adding
features, or weighting them by how distinctive they are, would separate items properly —
which is exactly what TF-IDF does in Week 4.

**2. Low ratings cannot express dislike.**
The profile is a sum of *rating × features*, and ratings are always positive, so a
1-star review still pushes a genre's weight **up**. User A rated *Love in Nairobi* 2/5,
yet Drama and Romance both ended up with a positive weight of 0.071, and *Broken Hearts*
was recommended to them third as a result. Centring ratings on the user's mean (so
below-average ratings become negative) would let the model learn what to avoid.

**3. Cold start: a new user with no history gets nothing.**
`recommend_movies` divides by `user_profile.sum()`, so a user with no ratings produces a
division by zero and a profile of `NaN`. Even one rating is not enough, as User C showed —
and with a single rating, normalization cancels the rating value entirely, so the model
cannot distinguish a 5-star from a 1-star reaction. The same applies to new *items*: a
film with missing genre metadata scores 0 for everyone and can never surface.

**4. Over-specialisation — the recommendations are too similar to what was already liked.**
By construction a movie can only score above zero on genres the user has already rated.
User B can *never* be shown *Space Warriors*, however much they might enjoy it, because
their Action and SciFi weights are 0. The model cannot produce serendipity, and it narrows
the user's exposure over time into a filter bubble.

**5. Rich item information is ignored.**
Actors, directors, release year, language, plot synopsis and reviews are all absent. Two
films with identical genre flags are indistinguishable to the model — *Future World* and
*Space Warriors* are literally the same item as far as the arithmetic is concerned, which is
exactly why they tie for first place in my own user's recommendations.

There is not even a column for **item format**. My unique entry, Money Heist, is a
multi-season television series, but the dataset cannot express that, so it is scored
identically to a 90-minute film. A recommender that cannot tell a two-hour movie from a
15-hour series is ignoring something most users would consider decisive.

**6. Garbage in, garbage out — the model inherits the metadata's flaws.**
The whole system rests on someone having tagged each film correctly and consistently. A
mislabelled or sparsely tagged film is mis-recommended to everyone, and because the genre
set is fixed and coarse, there is no way for the model to detect or recover from bad tags.
There is also no popularity or quality signal to fall back on when the metadata is thin.

## 18. Conclusion

*(Lab Step 28 workflow, plus the Final Question)*

### The content-based filtering workflow, as implemented

```
Item Metadata -> Item Profiles -> User Ratings -> User Profile
   -> Candidate Scores -> Remove Seen Items -> Rank Items -> Top-N Recommendations
```

Each stage maps to a section of this notebook: the item-feature matrix (section 4), the
rating-weighted and normalized user profile (section 7), the weighted dot product that
scores every candidate (section 8), the `isin` filter that drops seen items (section 9),
and the sort that produces Top-N (section 10).

### What I take away

A content-based recommender needs **no other users at all** — it works from item metadata
and one person's ratings, which makes it genuinely useful from a single user's first few
ratings and easy to explain ("recommended because you like Action and SciFi"). That
explainability is its real strength.

Its weaknesses, though, are structural rather than incidental. The model can only ever
recommend more of what the user has already chosen, it treats a low rating as weak
approval instead of disapproval, and with five binary features it produces so many tied
scores that the ranking is partly arbitrary. Richer features and term weighting (Week 4),
or a collaborative signal from other users, are needed to move past these.

---

### Final Question

> *If two users have watched exactly the same movies but gave them different ratings,
> should this recommender produce the same recommendations for both users?*

**No — and this model already behaves correctly in that respect.** The demonstration below
gives two users the *same four movies* (ids 1, 2, 3, 5) with different ratings.

In [36]:
same_movies = [1, 2, 3, 5]

user_x = pd.DataFrame({"movie_id": same_movies, "rating": [5, 2, 4, 5]})  # prefers Action/SciFi
user_y = pd.DataFrame({"movie_id": same_movies, "rating": [2, 5, 2, 2]})  # prefers the Drama/Romance film

print("User X profile:")
print(user_profile_for(movies, user_x, features).round(4), "\n")
print("User Y profile:")
print(user_profile_for(movies, user_y, features).round(4))

User X profile:
Action     0.5000
SciFi      0.3571
Drama      0.0714
Romance    0.0714
Comedy     0.0000
dtype: float64 

User Y profile:
Action     0.30
Drama      0.25
Romance    0.25
SciFi      0.20
Comedy     0.00
dtype: float64


In [37]:
# n=7 shows the full candidate list for both users, so the re-ordering is visible
# rather than hidden by a Top-3 cut-off.
print("User X full ranking:")
print(recommend_movies(movies, user_x, features, n=7)[["title", "recommendation_score"]]
      .round(4).to_string(index=False), "\n")
print("User Y full ranking:")
print(recommend_movies(movies, user_y, features, n=7)[["title", "recommendation_score"]]
      .round(4).to_string(index=False))

User X full ranking:
          title  recommendation_score
 Space Warriors                0.8571
    Money Heist                0.6429
City Detectives                0.5714
  Broken Hearts                0.1429
Wedding Weekend                0.0714
   Hidden Truth                0.0714
 Family Weekend                0.0000 

User Y full ranking:
          title  recommendation_score
    Money Heist                  0.80
City Detectives                  0.55
  Broken Hearts                  0.50
 Space Warriors                  0.50
Wedding Weekend                  0.25
   Hidden Truth                  0.25
 Family Weekend                  0.00


### Answer

**No, the recommender should not — and does not — produce the same recommendations.**

**Why, in terms of the Week 3 model.** Ratings enter the pipeline as the **weights** in
`rated_features.mul(user_ratings, axis=0)`. The user profile is a *rating-weighted* sum of
item profiles, so the ratings decide how much each watched movie contributes. Two users with
identical viewing histories but different ratings therefore build **different profiles**, and
since scoring is a dot product against that profile, they get different scores and a
different ranking. Only the *candidate set* is shared — the `isin` filter removes the same
four watched movies for both, leaving both users the same seven candidates.

**What the demonstration shows.** User X rated the Action/SciFi films highly; User Y rated
*Love in Nairobi* (Drama/Romance) highest and everything else low. The profiles diverge:

| Genre | User X | User Y |
|---|---|---|
| Action | **0.500** | 0.300 |
| SciFi | **0.357** | 0.200 |
| Drama | 0.071 | **0.250** |
| Romance | 0.071 | **0.250** |
| Comedy | 0.000 | 0.000 |

And the rankings genuinely reorder — including at the very top:

| Rank | User X | Score | User Y | Score |
|---|---|---|---|---|
| 1 | **Space Warriors** | **0.857** | **Money Heist** | **0.800** |
| 2 | Money Heist | 0.643 | City Detectives | 0.550 |
| 3 | City Detectives | 0.571 | Broken Hearts | 0.500 |
| 4 | Broken Hearts | 0.143 | **Space Warriors** | **0.500** |
| 5 | Wedding Weekend | 0.071 | Wedding Weekend | 0.250 |
| 6 | Hidden Truth | 0.071 | Hidden Truth | 0.250 |
| 7 | Family Weekend | 0.000 | Family Weekend | 0.000 |

The two users receive **different top recommendations** — *Space Warriors* for User X,
*Money Heist* for User Y — and *Space Warriors* itself falls from **1st (0.857) to 4th
(0.500)**. Every score differs between the two users. This is the desired behaviour: two
people who watched the same films but disagreed about them should not be told to watch the
same thing next.

**Two caveats worth stating.**

*First, an item with many features has a structural advantage.* Money Heist ranks 2nd for
User X and 1st for User Y — it is never far from the top for anybody, because carrying three
of the five features lets it collect weight from almost any profile. It loses to
*Space Warriors* for User X only because User X concentrates 0.857 of their total weight into
exactly the two genres *Space Warriors* has. That is a property of the scoring function
rather than evidence of a better match, and it means richly tagged items crowd out sparsely
tagged ones regardless of who is asking.

*Second, the model separates these users far too weakly.* User Y rated both Action films
2/5, yet **Action remains their single highest weight at 0.300.** Because ratings only ever
add positive weight, the model reads "rated it 2/5" as *mild approval of Action* rather than
dislike. A user who clearly preferred the romance still carries Action as their top
preference. So the model responds to rating differences in the right **direction** but at
nothing like the right **magnitude** — mean-centring the ratings, so below-average scores
become negative, is what would make these two users look as different as they actually are.